In [2]:
import torch
from typing import Any, Callable, Dict, List, Optional, Union
from diffusers import StableDiffusionPipeline
import random
import numpy as np
import matplotlib.pyplot as plt
from diffusers import StableDiffusionPipeline, DDIMScheduler
from tqdm import tqdm
from torch.nn.functional import cosine_similarity
from collections import defaultdict
from sklearn.cluster import KMeans
from transformers import CLIPTokenizer, CLIPTextModel
import open_clip
from open_clip.model import CustomTextCLIP

/home/nessessence/anaconda3/envs/uul/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
from gradient_surgery import AttentionGradientHook, generalize_gradient_projection

In [28]:
def get_learnable_parameters(model, train_method='esd-x'):
    learnable_params = []
    learnable_param_names = []
    for name, module in model.named_modules():
        if module.__class__.__name__ in ["Linear", "Conv2d", "LoRACompatibleLinear", "LoRACompatibleConv"]:
            if train_method == 'esd-x' and 'attn2' in name:
                for n, p in module.named_parameters():
                    learnable_param_names.append(name+'.'+n)
                    learnable_params.append(p)

            if train_method == 'esd-u' and ('attn2' not in name):
                for n, p in module.named_parameters():
                    learnable_param_names.append(name+'.'+n)
                    learnable_params.append(p)

            if train_method == 'esd-all' :
                for n, p in module.named_parameters():
                    learnable_param_names.append(name+'.'+n)
                    learnable_params.append(p)

            if train_method == 'esd-x-strict' and ('attn2.to_k' in name or 'attn2.to_v' in name):
                for n, p in module.named_parameters():
                    learnable_param_names.append(name+'.'+n)
                    learnable_params.append(p)

    return learnable_param_names, learnable_params

  
def install_hook_on_learnables(unet, hook_cls, learnable_param_names=None, **hook_kwargs):
    """
    Install `hook_cls` on selected attention processors.
      - If learnable_param_names is None → attach to ALL attention modules.
      - Otherwise → attach only to processors mapped from learnable_param_names.
    """
    if learnable_param_names is None:
        target_keys = set(unet.attn_processors.keys())
    else:
        # Note: append "processor" (no leading dot) because tags already end with "."
        target_keys = {
            f"{name[:name.index(tag) + len(tag)]}processor"
            for tag in (".attn1.", ".attn2.")
            for name in learnable_param_names
            if tag in name
        }

    new_map = {
        name: hook_cls(**hook_kwargs) if name in target_keys else proc
        for name, proc in unet.attn_processors.items()
    }
    unet.set_attn_processor(new_map)
    return sorted(target_keys)

# check if the hook is properly installed
def check_hook_installation(unet, hook_cls):
    count_total = 0
    count_hooked = 0
    for name, proc in unet.attn_processors.items():
        count_total += 1
        if isinstance(proc, hook_cls):
            count_hooked += 1
            print(f"✅ Hook installed at: {name}")
    print(f"\nTotal processors: {count_total}, Hooked: {count_hooked}")
    
    





In [30]:


DEVICE = "cuda:1" if torch.cuda.is_available() else "cpu"
MODEL_ID = "CompVis/stable-diffusion-v1-4"

# Load your pipeline
pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    safety_checker=None
).to(DEVICE)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe.set_progress_bar_config(disable=True)

unet = pipe.unet

Couldn't connect to the Hub: 500 Server Error: Internal Server Error for url: https://huggingface.co/api/models/CompVis/stable-diffusion-v1-4 (Request ID: Root=1-68f5f714-3f16b4ac1845a6974704a27d;68f6da47-7d26-4006-9e51-ae1be9ef9d08)

Internal Error - We're working hard to fix this as soon as possible!.
Will try to load from local cache.
Loading pipeline components...:  67%|██████████████████▋         | 4/6 [00:01<00:00,  3.98it/s]/home/nessessence/anaconda3/envs/uul/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Loading pipeline components...: 100%|████████████████████████████| 6/6 [00:01<00:00,  5.61it/s]
You have disabled the safety checker for <class 'diff

In [31]:
learnable_param_names, learnable_params = get_learnable_parameters(unet, train_method='esd-u')


target_keys = install_hook_on_learnables(unet, AttentionGradientHook, learnable_param_names=learnable_param_names)

print(target_keys)


check_hook_installation(unet, AttentionGradientHook)



['down_blocks.0.attentions.0.transformer_blocks.0.attn1.processor', 'down_blocks.0.attentions.1.transformer_blocks.0.attn1.processor', 'down_blocks.1.attentions.0.transformer_blocks.0.attn1.processor', 'down_blocks.1.attentions.1.transformer_blocks.0.attn1.processor', 'down_blocks.2.attentions.0.transformer_blocks.0.attn1.processor', 'down_blocks.2.attentions.1.transformer_blocks.0.attn1.processor', 'mid_block.attentions.0.transformer_blocks.0.attn1.processor', 'up_blocks.1.attentions.0.transformer_blocks.0.attn1.processor', 'up_blocks.1.attentions.1.transformer_blocks.0.attn1.processor', 'up_blocks.1.attentions.2.transformer_blocks.0.attn1.processor', 'up_blocks.2.attentions.0.transformer_blocks.0.attn1.processor', 'up_blocks.2.attentions.1.transformer_blocks.0.attn1.processor', 'up_blocks.2.attentions.2.transformer_blocks.0.attn1.processor', 'up_blocks.3.attentions.0.transformer_blocks.0.attn1.processor', 'up_blocks.3.attentions.1.transformer_blocks.0.attn1.processor', 'up_blocks.3.a

['down_blocks.0.attentions.0.transformer_blocks.0.attn2.processor',
 'down_blocks.0.attentions.1.transformer_blocks.0.attn2.processor',
 'down_blocks.1.attentions.0.transformer_blocks.0.attn2.processor',
 'down_blocks.1.attentions.1.transformer_blocks.0.attn2.processor',
 'down_blocks.2.attentions.0.transformer_blocks.0.attn2.processor',
 'down_blocks.2.attentions.1.transformer_blocks.0.attn2.processor',
 'mid_block.attentions.0.transformer_blocks.0.attn2.processor',
 'up_blocks.1.attentions.0.transformer_blocks.0.attn2.processor',
 'up_blocks.1.attentions.1.transformer_blocks.0.attn2.processor',
 'up_blocks.1.attentions.2.transformer_blocks.0.attn2.processor',
 'up_blocks.2.attentions.0.transformer_blocks.0.attn2.processor',
 'up_blocks.2.attentions.1.transformer_blocks.0.attn2.processor',
 'up_blocks.2.attentions.2.transformer_blocks.0.attn2.processor',
 'up_blocks.3.attentions.0.transformer_blocks.0.attn2.processor',
 'up_blocks.3.attentions.1.transformer_blocks.0.attn2.processor',


In [8]:
# install hook processor for this step
hook = AttentionGradientHook()
unet.set_attn_processor(hook)

✅ Hook installed at: down_blocks.0.attentions.0.transformer_blocks.0.attn1.processor
✅ Hook installed at: down_blocks.0.attentions.0.transformer_blocks.0.attn2.processor
✅ Hook installed at: down_blocks.0.attentions.1.transformer_blocks.0.attn1.processor
✅ Hook installed at: down_blocks.0.attentions.1.transformer_blocks.0.attn2.processor
✅ Hook installed at: down_blocks.1.attentions.0.transformer_blocks.0.attn1.processor
✅ Hook installed at: down_blocks.1.attentions.0.transformer_blocks.0.attn2.processor
✅ Hook installed at: down_blocks.1.attentions.1.transformer_blocks.0.attn1.processor
✅ Hook installed at: down_blocks.1.attentions.1.transformer_blocks.0.attn2.processor
✅ Hook installed at: down_blocks.2.attentions.0.transformer_blocks.0.attn1.processor
✅ Hook installed at: down_blocks.2.attentions.0.transformer_blocks.0.attn2.processor
✅ Hook installed at: down_blocks.2.attentions.1.transformer_blocks.0.attn1.processor
✅ Hook installed at: down_blocks.2.attentions.1.transformer_block